# MILP v10 — Bridge Comparison: v1 (spec v3) vs v7

Runs the MILP v10 mainline case (M2_I1_R0) with both bridge packages,
then compares the resulting optimal sizing and costs.

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from milp_common import get_config, load_data, CASE_TABLE, format_results, get_monthly_basic_charge

import gurobipy as gp
from gurobipy import GRB
import time

print('Imports OK')

Imports OK


In [2]:
def build_and_solve(CFG, repday_data, calendar_order, info, case_flags):
    """Build and solve the v10 MILP (spec v2.3) — same model as milp_v10_cases."""
    use_method1 = case_flags.get('method1', True) and calendar_order is not None
    n_repdays = info['n_repdays']
    n_scenarios = info['n_scenarios']
    n_hours = info['n_hours']
    N = n_repdays * n_scenarios * n_hours
    n_seg = len(CFG['pwl_deg_lambda_k'])
    PV_RATING = CFG['pv_rating_kw']
    PV_FIXED = CFG['pv_fixed_kw']

    pv_arr = np.empty(N); load_arr = np.empty(N)
    pw_arr = np.empty(N); tou_arr = np.empty(N); month_arr = np.empty(N, dtype=int)

    def flat(d, s, t): return d * (n_scenarios * n_hours) + s * n_hours + t

    for d in range(n_repdays):
        dd = repday_data[d]
        for s in range(n_scenarios):
            sc = dd['scenarios'][s]
            start = flat(d, s, 0)
            sl = slice(start, start + n_hours)
            pv_arr[sl] = sc['pv_kw'] / PV_RATING * PV_FIXED
            load_arr[sl] = sc['load_kw']
            pw_arr[sl] = sc['prob'] * dd['weight']
            tou_arr[sl] = dd['tou']
            month_arr[sl] = dd['month']

    is_first = np.zeros(N, dtype=bool)
    for d in range(n_repdays):
        for s in range(n_scenarios):
            is_first[flat(d, s, 0)] = True
    first_hours = np.where(is_first)[0]
    later_hours = np.where(~is_first)[0]
    prev_idx_arr = np.arange(N) - 1

    b_k = CFG['pwl_deg_b_k']
    lam_k = CFG['pwl_deg_lambda_k']

    m = gp.Model('milp_v10')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = CFG['time_limit']
    m.Params.Method = 2; m.Params.Presolve = 2; m.Params.Cuts = 2
    m.Params.MIPGap = 1e-4; m.Params.MIPFocus = 1

    # PV is fixed — not a decision variable
    cap_bp = m.addVar(ub=CFG['bess_p_max_kw'], name='P_B')
    cap_be = m.addVar(ub=CFG['bess_e_max_kwh'], name='E_B')
    cap_cd = m.addVar(name='CC')

    p_grid = m.addMVar(N, name='P_grid'); p_pv_load = m.addMVar(N, name='P_pv_load')
    p_pv_ch = m.addMVar(N, name='P_pv_ch'); p_curt = m.addMVar(N, name='P_curt')
    p_ch = m.addMVar(N, name='P_ch'); p_dis = m.addMVar(N, name='P_dis')
    u_bin = m.addMVar(N, vtype=GRB.BINARY, name='u')
    p_ch_g = m.addMVar(N, name='P_ch_g'); p_dis_g = m.addMVar(N, name='P_dis_g')
    delta_e = m.addMVar(N, lb=-GRB.INFINITY, name='dE')
    e_dis_seg = m.addMVar((N, n_seg), name='e_dis_seg')
    trec_buy = m.addVar(name='E_TREC')
    m.update()

    eff_ch = CFG['eff_charge']; eff_dis_inv = 1.0 / CFG['eff_discharge']

    # C1-C2
    m.addConstrs((p_grid[i] + p_pv_load[i] + p_dis[i] == load_arr[i] + p_ch[i] for i in range(N)), name='C1')
    m.addConstrs((p_pv_load[i] + p_pv_ch[i] + p_curt[i] == pv_arr[i] for i in range(N)), name='C2')
    m.addConstrs((p_pv_ch[i] <= p_ch[i] for i in range(N)), name='C2b')
    # C3-C4
    m.addConstrs((p_ch[i] <= cap_bp * u_bin[i] for i in range(N)), name='C3_ch')
    m.addConstrs((p_dis[i] <= cap_bp * (1 - u_bin[i]) for i in range(N)), name='C3_dis')
    # C5
    m.addConstrs((delta_e[i] == eff_ch * p_ch[i] - eff_dis_inv * p_dis[i] for i in first_hours), name='C5i')
    m.addConstrs((delta_e[i] == delta_e[prev_idx_arr[i]] + eff_ch * p_ch[i] - eff_dis_inv * p_dis[i] for i in later_hours), name='C5c')

    if use_method1:
        n_cal = len(calendar_order)
        E_inter = m.addMVar(n_cal, lb=0, name='E_inter')
        for ci, cd in enumerate(calendar_order):
            d = cd['d_idx']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    fi = flat(d, s, t)
                    m.addConstr(E_inter[ci] + delta_e[fi] >= CFG['soc_min'] * cap_be)
                    m.addConstr(E_inter[ci] + delta_e[fi] <= CFG['soc_max'] * cap_be)
        for ci in range(n_cal - 1):
            d = calendar_order[ci]['d_idx']
            ede = gp.LinExpr()
            for s in range(n_scenarios):
                ede += repday_data[d]['scenarios'][s]['prob'] * delta_e[flat(d, s, n_hours-1)]
            m.addConstr(E_inter[ci+1] == E_inter[ci] + ede)
        m.addConstr(E_inter[0] == CFG['soc_init'] * cap_be)
        dl = calendar_order[-1]['d_idx']
        edl = gp.LinExpr()
        for s in range(n_scenarios):
            edl += repday_data[dl]['scenarios'][s]['prob'] * delta_e[flat(dl, s, n_hours-1)]
        eps = CFG['epsilon_term']
        m.addConstr(E_inter[n_cal-1] + edl - E_inter[0] <= eps * cap_be)
        m.addConstr(E_inter[n_cal-1] + edl - E_inter[0] >= -eps * cap_be)

        E_g_inter = m.addMVar(n_cal, lb=0, name='E_g_inter')
        delta_e_g = m.addMVar(N, lb=-GRB.INFINITY, name='dE_g')
        m.addConstrs((p_ch_g[i] <= p_pv_ch[i] for i in range(N)), name='C13cg')
        m.addConstrs((p_dis_g[i] <= p_dis[i] for i in range(N)), name='C13dg')
        m.addConstrs((delta_e_g[i] == eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i] for i in first_hours), name='C13ai')
        m.addConstrs((delta_e_g[i] == delta_e_g[prev_idx_arr[i]] + eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i] for i in later_hours), name='C13ac')
        for ci, cd in enumerate(calendar_order):
            d = cd['d_idx']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    fi = flat(d, s, t)
                    m.addConstr(E_g_inter[ci] + delta_e_g[fi] >= 0)
                    m.addConstr(E_g_inter[ci] + delta_e_g[fi] <= E_inter[ci] + delta_e[fi])
        for ci in range(n_cal - 1):
            d = calendar_order[ci]['d_idx']
            edg = gp.LinExpr()
            for s in range(n_scenarios):
                edg += repday_data[d]['scenarios'][s]['prob'] * delta_e_g[flat(d, s, n_hours-1)]
            m.addConstr(E_g_inter[ci+1] == E_g_inter[ci] + edg)
        m.addConstr(E_g_inter[0] == 0)
        edgl = gp.LinExpr()
        for s in range(n_scenarios):
            edgl += repday_data[dl]['scenarios'][s]['prob'] * delta_e_g[flat(dl, s, n_hours-1)]
        m.addConstr(E_g_inter[n_cal-1] + edgl <= CFG['epsilon_g_term'] * cap_be)
        m.addConstr(E_g_inter[n_cal-1] + edgl >= -CFG['epsilon_g_term'] * cap_be)
    else:
        for i in range(N):
            m.addConstr(CFG['soc_init'] * cap_be + delta_e[i] >= CFG['soc_min'] * cap_be)
            m.addConstr(CFG['soc_init'] * cap_be + delta_e[i] <= CFG['soc_max'] * cap_be)
        delta_e_g = m.addMVar(N, lb=-GRB.INFINITY, name='dE_g')
        m.addConstrs((p_ch_g[i] <= p_pv_ch[i] for i in range(N)), name='C13cg')
        m.addConstrs((p_dis_g[i] <= p_dis[i] for i in range(N)), name='C13dg')
        m.addConstrs((delta_e_g[i] == eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i] for i in first_hours), name='C13ai')
        m.addConstrs((delta_e_g[i] == delta_e_g[prev_idx_arr[i]] + eff_ch * p_ch_g[i] - eff_dis_inv * p_dis_g[i] for i in later_hours), name='C13ac')
        for i in range(N):
            m.addConstr(delta_e_g[i] >= 0)
            m.addConstr(delta_e_g[i] <= CFG['soc_init'] * cap_be + delta_e[i])

    # C10-C11
    all_months = list(range(1, 13))
    if use_method1:
        D_max = {mo: m.addVar(name=f'Dm{mo}') for mo in all_months}
        kappa = CFG['kappa']
        for ci, cd in enumerate(calendar_order):
            d = cd['d_idx']; mo = cd['month_id']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    m.addConstr(D_max[mo] >= kappa * p_grid[flat(d, s, t)])
    else:
        months_in = sorted(set(month_arr))
        D_max = {mo: m.addVar(name=f'Dm{mo}') for mo in months_in}
        all_months = months_in
        kappa = CFG['kappa']
        for i in range(N):
            m.addConstr(D_max[int(month_arr[i])] >= kappa * p_grid[i])

    oc1 = {mo: m.addVar(name=f'o1{mo}') for mo in all_months}
    oc2 = {mo: m.addVar(name=f'o2{mo}') for mo in all_months}
    for mo in all_months:
        m.addConstr(oc1[mo] >= D_max[mo] - cap_cd)
        m.addConstr(oc1[mo] <= 0.10 * cap_cd)
        m.addConstr(oc2[mo] >= D_max[mo] - 1.10 * cap_cd)

    # C12 RE20
    total_load = float(np.sum(pw_arr * load_arr))
    re_expr = gp.quicksum(pw_arr[i] * (p_pv_load[i] + p_dis_g[i]) for i in range(N))
    m.addConstr(re_expr + trec_buy >= CFG['re_target'] * total_load, name='RE20')

    # C14 PWL degradation
    for i in range(N):
        m.addConstr(gp.quicksum(e_dis_seg[i, k] for k in range(n_seg)) == p_dis[i])
        for k in range(n_seg):
            m.addConstr(e_dis_seg[i, k] <= (b_k[k+1] - b_k[k]) * cap_be)

    # Objective
    pv_annuity = PV_FIXED * CFG['capex_pv_per_kw'] * CFG['crf_pv']
    AEC_inv = (pv_annuity
        + cap_bp * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
        + cap_be * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess']
                    + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate']))
    AEC_ene = gp.quicksum(pw_arr[i] * p_grid[i] * float(tou_arr[i]) for i in range(N))
    AEC_basic = gp.LinExpr()
    for mo in all_months:
        AEC_basic += cap_cd * get_monthly_basic_charge(mo, CFG)
    AEC_over = gp.LinExpr()
    for mo in all_months:
        rate = get_monthly_basic_charge(mo, CFG)
        AEC_over += oc1[mo] * rate * CFG['oc_within_10pct_mult'] + oc2[mo] * rate * CFG['oc_beyond_10pct_mult']
    AEC_green = CFG['trec_cost_per_kwh'] * trec_buy
    if use_method1:
        n_d = {}
        for cd in calendar_order: n_d[cd['d_idx']] = n_d.get(cd['d_idx'], 0) + 1
        AEC_deg = gp.LinExpr()
        for d in range(n_repdays):
            mult = n_d.get(d, repday_data[d]['weight'])
            for s in range(n_scenarios):
                ps = repday_data[d]['scenarios'][s]['prob']
                for t in range(n_hours):
                    fi = flat(d, s, t)
                    for k in range(n_seg):
                        AEC_deg += mult * ps * lam_k[k] * e_dis_seg[fi, k]
    else:
        AEC_deg = gp.LinExpr()
        for i in range(N):
            for k in range(n_seg):
                AEC_deg += pw_arr[i] * lam_k[k] * e_dis_seg[i, k]

    m.setObjective(AEC_inv + AEC_ene + AEC_basic + AEC_over + AEC_green + AEC_deg, GRB.MINIMIZE)

    t0 = time.time()
    m.optimize()
    st = time.time() - t0
    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL): return None

    re_val = sum(pw_arr[i] * (p_pv_load[i].X + p_dis_g[i].X) for i in range(N))
    cost_breakdown = {
        'AEC_inv': AEC_inv.getValue(), 'AEC_ene': AEC_ene.getValue(),
        'AEC_basic': AEC_basic.getValue(), 'AEC_over': AEC_over.getValue(),
        'AEC_green': AEC_green.getValue(), 'AEC_deg': AEC_deg.getValue(),
    }
    return {
        'cap_pv': PV_FIXED, 'cap_bp': cap_bp.X, 'cap_be': cap_be.X, 'cap_cd': cap_cd.X,
        'obj': m.ObjVal, 're_pct': re_val / total_load * 100,
        'trec_cost': trec_buy.X * CFG['trec_cost_per_kwh'], 'solve_time': st, 'gap': m.MIPGap,
        'cost_breakdown': cost_breakdown,
    }

print('Model builder loaded')

Model builder loaded


## Run Mainline (M2_I1_R0) with Both Bridge Packages

In [3]:
mainline = {'method1': True, 'risk_days': True, 'prob_pv': True, 'uplift': None}

# ── Bridge v1 (old: 95 repdays, 20 body + 75 risk) ──
print('='*60)
print('  Bridge v1 (spec v3): 20 body + 75 risk = 95 repdays')
print('='*60)
CFG_v1 = get_config()
CFG_v1['bridge_dir'] = '../bridge_outputs_v1'
rd_v1, co_v1, info_v1 = load_data(CFG_v1, mainline)
res_v1 = build_and_solve(CFG_v1, rd_v1, co_v1, info_v1, mainline)
print(f'  Result: PV={res_v1["cap_pv"]:,.0f} kW, BESS={res_v1["cap_bp"]:,.0f}/{res_v1["cap_be"]:,.0f}, '
      f'Contract={res_v1["cap_cd"]:,.0f}, Cost={res_v1["obj"]/1e6:.2f}M, RE={res_v1["re_pct"]:.1f}%')

# ── Bridge v7 (new: 44 repdays, 16 body + 28 risk) ──
print('\n' + '='*60)
print('  Bridge v7: 16 body + 28 risk = 44 repdays')
print('='*60)
CFG_v7 = get_config()
CFG_v7['bridge_dir'] = '../bridge_outputs'
rd_v7, co_v7, info_v7 = load_data(CFG_v7, mainline)
res_v7 = build_and_solve(CFG_v7, rd_v7, co_v7, info_v7, mainline)
print(f'  Result: PV={res_v7["cap_pv"]:,.0f} kW, BESS={res_v7["cap_bp"]:,.0f}/{res_v7["cap_be"]:,.0f}, '
      f'Contract={res_v7["cap_cd"]:,.0f}, Cost={res_v7["obj"]/1e6:.2f}M, RE={res_v7["re_pct"]:.1f}%')

  Bridge v1 (spec v3): 20 body + 75 risk = 95 repdays
CRF PV (20yr, r=0.05): 0.0802
CRF BESS (15yr, r=0.05): 0.0963
  Repdays: 95 (20 body + 75 risk)
  Scenarios: 5/repday, 24 hours
  Weight sum: 365
  Calendar days: 365
Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2797924


Academic license 2797924 - for non-commercial use only - registered to m1___@mail.ntust.edu.tw


  Result: PV=2,687 kW, BESS=1,321/9,292, Contract=3,252, Cost=110.33M, RE=14.5%

  Bridge v7: 16 body + 28 risk = 44 repdays
CRF PV (20yr, r=0.05): 0.0802
CRF BESS (15yr, r=0.05): 0.0963
  Repdays: 44 (16 body + 28 risk)
  Scenarios: 5/repday, 24 hours
  Weight sum: 365
  Calendar days: 365


  Result: PV=2,687 kW, BESS=825/4,321, Contract=3,586, Cost=107.82M, RE=14.6%


In [4]:
# ── Comparison Table ──
comp = pd.DataFrame([
    format_results('Bridge_v1 (95 repdays)', res_v1['cap_pv'], res_v1['cap_bp'], res_v1['cap_be'],
                   res_v1['cap_cd'], res_v1['obj'], res_v1['re_pct'], res_v1['trec_cost'],
                   res_v1['solve_time'], res_v1['cost_breakdown'], CFG_v1),
    format_results('Bridge_v7 (44 repdays)', res_v7['cap_pv'], res_v7['cap_bp'], res_v7['cap_be'],
                   res_v7['cap_cd'], res_v7['obj'], res_v7['re_pct'], res_v7['trec_cost'],
                   res_v7['solve_time'], res_v7['cost_breakdown'], CFG_v7),
])
print(comp.to_string(index=False))
comp.to_csv('../milp_outputs/bridge_comparison.csv', index=False)
print('\nSaved to milp_outputs/bridge_comparison.csv')

                  case  pv_kw  bess_p_kw  bess_e_kwh  ep_ratio  contract_kw  total_cost_M  AEC_inv_M  AEC_ene_M  AEC_basic_M  AEC_over_M  AEC_green_M  AEC_deg_M  trec_M  re_pct  solve_s
Bridge_v1 (95 repdays)   2687     1321.3      9291.5       7.0       3252.1        110.33      17.79      76.44         7.62        0.33         5.45       2.70    5.45    14.5     32.9
Bridge_v7 (44 repdays)   2687      825.3      4321.0       5.2       3585.9        107.82      13.13      79.24         8.40        0.50         5.34       1.22    5.34    14.6     24.3

Saved to milp_outputs/bridge_comparison.csv
